In [2]:
import json

In [14]:
import os

In [17]:
%pip install pillow

  Using cached pillow-12.2.0-cp311-cp311-win_amd64.whl.metadata (9.0 kB)
Using cached pillow-12.2.0-cp311-cp311-win_amd64.whl (7.1 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from PIL import Image

In [48]:
def convert_to_ls_format(data_dir):
    annot_dir = data_dir + 'annotations/'
    img_dir = data_dir + 'images/'

    all_annotations = []

    for i, filename in enumerate(os.listdir(annot_dir)):
        task_annot = {
            'id': filename[:-5],
            'data': {
                'image': f'/data/local-files/?d={data_dir}images/{filename[:-5]}.png',
                'image_path': f'{data_dir}images/{filename[:-5]}.png'
            },
            'predictions': [
                {
                    'model_version': 'funsd-export',
                    'result': []
                }
            ]
        }

        with open(annot_dir + filename, encoding='utf-8') as f:
            json_loaded = json.load(f)
        
        with Image.open(img_dir + filename[:-5] + '.png') as img:
            img_w, img_h = img.size
        
        for form in json_loaded['form']:
            x_left, y_top, x_right, y_bottom = form['box']

            x = (x_left / img_w) * 100
            y = (y_top / img_h) * 100
            width = ((x_right - x_left) / img_w) * 100
            height = ((y_bottom - y_top) / img_h) * 100

            rectanglelabels = [form['label']]

            result = {
                'original_width': img_w,
                'original_height': img_h,
                'value': {
                            'x': x,
                            'y': y,
                            'width': width,
                            'height': height,
                            'rotation': 0,
                            'rectanglelabels': rectanglelabels
                        },
                    # 'id': 'XXX',
                    'from_name': 'label',
                    'to_name': 'image',
                    'type': 'rectanglelabels',
                    # 'origin': 'XXX'
            }

            task_annot['predictions'][0]['result'].append(result)
        
        all_annotations.append(task_annot)
    
    return all_annotations

In [49]:
testing_annot = convert_to_ls_format('dataset/testing_data/')

with open('testing_annot.json', 'w', encoding='utf-8') as f:
    json.dump(testing_annot, f, ensure_ascii=False, indent=4)

In [50]:
training_annot = convert_to_ls_format('dataset/training_data/')

with open('training_annot.json', 'w', encoding='utf-8') as f:
    json.dump(training_annot, f, ensure_ascii=False, indent=4)